

# Lab 3 Solution Walkthrough: I2S Audio Transmitter

**Lecture Date:** Friday, October 9, 2026  
**Simulation Due:** Wednesday, October 14, 2026  
**Hardware Validation & Teach-Back:** Tuesday, October 20, 2026 (during Lab 4 meeting)

This notebook walks through the complete reference solution for Lab 3. The goal is to understand how a formal hardware contract translates into modular SystemVerilog RTL, mathematically verifiable golden vectors, and a self-checking testbench.

---

## 1. Hardware Contract

The FPGA acts as the **I2S Master**, streaming audio/SDR data to a Raspberry Pi Pico (RP2040) acting as the **I2S Slave**.

* **Master Clock ($f_{\text{clk}}$):** $30.720\text{ MHz}$
* **Bit Clock ($\text{BCK}$):** Toggles every $5$ master-clock cycles $\to 3.072\text{ MHz}$ square wave ($10$ master-clock cycles per bit period).
* **Word Select ($\text{WS}$ / $\text{LRCK}$):** Runs at $48\text{ kHz}$ ($64\text{ BCK}$ periods per audio frame).
  * $\text{WS} = 0$: Left channel / $I(t)$ data slot ($32\text{ BCK}$ periods).
  * $\text{WS} = 1$: Right channel / $Q(t)$ data slot ($32\text{ BCK}$ periods).
* **Slot Data Format:** 24-bit two's complement audio, MSB-first, padded with 8 trailing zeros to fill the 32-bit slot:
  $$\text{Slot Word} = \{\text{sample}[23:0], 8\text{'b}0\}$$
* **Edge Timing Contract:**
  * **Transmitter (FPGA):** Updates $\text{WS}$ and drives new serial data bits on **$\text{BCK}$ falling edges**.
  * **Receiver (Pico):** Samples $\text{WS}$ and data on **$\text{BCK}$ rising edges**.
  * The transmitted bit must be completely stable prior to the rising edge that samples it.

---

## 2. Test Stimulus: The 24-Bit Sawtooth Wave

To verify the audio pipeline end-to-end, the FPGA generates a 24-bit signed sawtooth wave on both channels.

```text
Amplitude (+1.0 FS)  -->     /     /|
                            / |   / |
Center (0 / Silence) ------/--|--/--|-(0x000000)
                          /   | /   |
Amplitude (-1.0 FS)  --> /    |/    | (0x800000 = SAW_START)
                             Trough    Trough
```

### Two's Complement Interpretation
* **`24'h800000` ($-8,388,608$):** Most negative signed 24-bit value (the waveform trough).
* **`24'h000000` ($0$):** Midpoint (zero-crossing / silence).
* **`24'h7FFFFF` ($+8,388,607$):** Peak positive value.
* **`SAW_STEP = 24'h010000` ($+65,536$):** The step added once per audio frame ($48\text{ kHz}$).
* **Natural Roll-Over:** Adding `SAW_STEP` past `24'h7FFFFF` causes the 24-bit accumulator to overflow back to `24'h800000`, producing the sharp cliff of the sawtooth wave without any reset logic.
* **Wave Frequency:** $\frac{2^{24}}{2^{16}} = 256\text{ samples/cycle} \implies \frac{48,000\text{ Hz}}{256} = 187.5\text{ Hz}$ (audible tone).

---

## 3. Generating "Golden Vectors" (Python)

We use Python to generate the exact sequence of 32-bit words that the Pico receiver should sample across the first 3 frames (6 slots). The script prints the bit patterns and saves `expected_words.mem` for the SystemVerilog testbench.

```python
from pathlib import Path

FRAME_COUNT = 3
SAW_START = 0x800000
SAW_STEP = 0x010000
SAMPLE_MASK = (1 << 24) - 1

def slot_word(sample):
    # Pack 24-bit sample into MSBs of 32-bit slot word
    return (sample & SAMPLE_MASK) << 8

words = []
text_lines = [
    "# Golden vectors: 3 frames (6 slots total)",
    "# Format: exactly 32 bits sampled on BCK rising edges per slot"
]

sample = SAW_START
for frame_idx in range(FRAME_COUNT):
    word = slot_word(sample)
    bits = f"{word:032b}"
    text_lines.append(f"frame={frame_idx} sample=0x{sample:06x} ws=0 (Left)  bits={bits}")
    text_lines.append(f"frame={frame_idx} sample=0x{sample:06x} ws=1 (Right) bits={bits}")
    # Both Left and Right slots carry the same sample in this demo
    words.extend([f"{word:08x}", f"{word:08x}"])
    sample = (sample + SAW_STEP) & SAMPLE_MASK

# Save to expected_words.mem
mem_path = Path("expected_words.mem")
mem_path.write_text("\n".join(words) + "\n")

print("\n".join(text_lines))
print(f"\nWritten to {mem_path.resolve()}:")
print("\n".join(words))
```

---

## 4. RTL Implementation (SystemVerilog)

We follow the principle of **Separation of Concerns**:
1. **`sawtooth_gen`**: Computes the wave samples; advances only when requested.
2. **`i2s_master_tx`**: Drives $\text{BCK}$, $\text{WS}$, and serializes the 32-bit slots using a clean 0–63 bit counter.
3. **`i2s_tx_demo`**: Top-level module wiring the generator to the transmitter.

Save the following code as **`i2s_tx_demo.sv`**:

```systemverilog
`timescale 1ns/1ps

// ============================================================================
// Submodule: sawtooth_gen
// ============================================================================
module sawtooth_gen #(
    parameter logic [23:0] SAW_START = 24'h800000,
    parameter logic [23:0] SAW_STEP  = 24'h010000
) (
    input  logic        clk_30m,
    input  logic        reset_n,
    input  logic        sample_req,
    output logic [23:0] out_data
);

    always_ff @(posedge clk_30m or negedge reset_n) begin
        if (!reset_n) begin
            out_data <= SAW_START;
        end else if (sample_req) begin
            out_data <= out_data + SAW_STEP;
        end
    end

endmodule

// ============================================================================
// Submodule: i2s_master_tx
// ============================================================================
module i2s_master_tx #(
    parameter integer BCK_HALF_DIV = 5
) (
    input  logic        clk_30m,
    input  logic        reset_n,
    input  logic [23:0] left_data,
    input  logic [23:0] right_data,
    output logic        sample_req,
    output logic        i2s_bck,
    output logic        i2s_ws,
    output logic        i2s_dout
);

    logic [2:0] bck_divider;
    logic [5:0] frame_bit;

    wire [31:0] left_slot  = {left_data,  8'b0};
    wire [31:0] right_slot = {right_data, 8'b0};

    always_ff @(posedge clk_30m or negedge reset_n) begin
        if (!reset_n) begin
            bck_divider <= '0;
            frame_bit   <= 6'd0;
            i2s_bck     <= 1'b0;
            i2s_ws      <= 1'b0;
            i2s_dout    <= 1'b0;
            sample_req  <= 1'b0;
        end else begin
            sample_req <= 1'b0; // Default pulse state

            if (bck_divider == BCK_HALF_DIV - 1) begin
                bck_divider <= '0;
                i2s_bck     <= ~i2s_bck;

                // Transmit on the FALLING edge of BCK
                if (i2s_bck == 1'b1) begin
                    // 1. Advance the 64-bit frame counter
                    if (frame_bit == 6'd63) begin
                        frame_bit  <= 6'd0;
                        sample_req <= 1'b1; // Trigger sample update for next frame
                    end else begin
                        frame_bit <= frame_bit + 1'b1;
                    end

                    // 2. Drive Word Select (WS)
                    // frame_bit 0..31: Left (WS=0); frame_bit 32..63: Right (WS=1)
                    if (frame_bit <= 6'd31) begin
                        i2s_ws   <= 1'b0;
                        i2s_dout <= left_slot[6'd31 - frame_bit];
                    end else begin
                        i2s_ws   <= 1'b1;
                        i2s_dout <= right_slot[6'd63 - frame_bit];
                    end
                end
            end else begin
                bck_divider <= bck_divider + 1'b1;
            end
        end
    end

endmodule

// ============================================================================
// Top-Level Module: i2s_tx_demo
// ============================================================================
module i2s_tx_demo #(
    parameter integer BCK_HALF_DIV = 5,
    parameter logic [23:0] SAW_START = 24'h800000,
    parameter logic [23:0] SAW_STEP  = 24'h010000
) (
    input  logic clk_30m,
    input  logic reset_n,
    output logic i2s_bck,
    output logic i2s_ws,
    output logic i2s_rx_data // Named i2s_rx_data to match top-level pinout
);

    logic [23:0] sample_data;
    logic        sample_tick;

    sawtooth_gen #(
        .SAW_START(SAW_START),
        .SAW_STEP(SAW_STEP)
    ) u_sawtooth (
        .clk_30m   (clk_30m),
        .reset_n   (reset_n),
        .sample_req(sample_tick),
        .out_data  (sample_data)
    );

    i2s_master_tx #(
        .BCK_HALF_DIV(BCK_HALF_DIV)
    ) u_i2s_tx (
        .clk_30m   (clk_30m),
        .reset_n   (reset_n),
        .left_data (sample_data),
        .right_data(sample_data),
        .sample_req(sample_tick),
        .i2s_bck   (i2s_bck),
        .i2s_ws    (i2s_ws),
        .i2s_dout  (i2s_rx_data)
    );

endmodule
```

---

## 5. Self-Checking Testbench (`i2s_tb.sv`)

The testbench acts as the **Pico receiver model**:
1. Observes each `posedge i2s_bck`.
2. Validates that `i2s_ws` matches the expected slot level ($0$ for Left, $1$ for Right).
3. Shifts in exactly 32 serial data bits per slot.
4. Compares each completed 32-bit word against `expected_words.mem`.
5. Halts with `PASS` after 6 verified slots (3 frames), or `FAIL` if any bit deviates.

Save this code as **`i2s_tb.sv`**:

```systemverilog
`timescale 1ns/1ps

module i2s_tb;

    // 1. Master Clock Generation (30.720 MHz -> Period ~32.552 ns)
    logic clk_30m = 1'b0;
    always #16.276 clk_30m = ~clk_30m;

    logic reset_n;
    logic i2s_bck;
    logic i2s_ws;
    logic i2s_rx_data;

    // 2. Device Under Test (DUT)
    i2s_tx_demo dut (
        .clk_30m    (clk_30m),
        .reset_n    (reset_n),
        .i2s_bck    (i2s_bck),
        .i2s_ws     (i2s_ws),
        .i2s_rx_data(i2s_rx_data)
    );

    // 3. Golden Vector Memory
    logic [31:0] expected_words [0:5];
    initial $readmemh("expected_words.mem", expected_words);

    // 4. Verification Engine
    integer slot_idx;
    integer bit_idx;
    logic [31:0] captured_word;
    logic expected_ws;

    initial begin
        // Reset sequence
        reset_n = 1'b0;
        #100;
        @(negedge clk_30m);
        reset_n = 1'b1;

        // Verify 6 slots (3 frames)
        for (slot_idx = 0; slot_idx < 6; slot_idx++) begin
            expected_ws   = (slot_idx % 2 == 1); // 0 = Left, 1 = Right
            captured_word = 32'h0;

            for (bit_idx = 0; bit_idx < 32; bit_idx++) begin
                @(posedge i2s_bck);

                // Check WS level on each sampling edge
                if (i2s_ws !== expected_ws) begin
                    $display("FAIL: WS mismatch at slot %0d, bit %0d. Expected %b, got %b",
                             slot_idx, bit_idx, expected_ws, i2s_ws);
                    $finish;
                end

                // Shift in data bit (MSB-first)
                captured_word = {captured_word[30:0], i2s_rx_data};
            end

            // Compare word against golden memory
            if (captured_word !== expected_words[slot_idx]) begin
                $display("FAIL: Data mismatch at slot %0d (WS=%b). Expected 0x%08x, got 0x%08x",
                         slot_idx, expected_ws, expected_words[slot_idx], captured_word);
                $finish;
            end else begin
                $display("Slot %0d (WS=%b) MATCHED: 0x%08x",
                         slot_idx, expected_ws, captured_word);
            end
        end

        $display("\n==============================");
        $display("   SIMULATION PASSED (PASS)");
        $display("==============================\n");
        $finish;
    end

    // Watchdog timeout
    initial begin
        #500000;
        $display("FAIL: Simulation timeout expired.");
        $finish;
    end

endmodule
```

---

## 6. Simulation & The Negative Test

### Running the Positive Verification Test
Execute the following cell to compile and run the reference design with Icarus Verilog:

```python
import subprocess
from pathlib import Path

commands = [
    ["iverilog", "-g2012", "-o", "i2s_pass.vvp", "i2s_tx_demo.sv", "i2s_tb.sv"],
    ["vvp", "i2s_pass.vvp"]
]

for cmd in commands:
    print("$", " ".join(cmd))
    res = subprocess.run(cmd, text=True, capture_output=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr)
    if res.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
```

### The Negative Test (Catching Subtle Bugs)
A waveform might